In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

In [2]:
DATA_DIR = Path("../docs")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "alpha-guide-rag"

In [3]:
loader = PyPDFDirectoryLoader(DATA_DIR)
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} documents from {DATA_DIR}")

Loaded 312 documents from ..\docs


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(raw_docs)

In [10]:
import os
os.environ["HTTPS_PROXY"] = os.getenv("HTTP_PROXY")
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY")

In [42]:
import os

PROXY = "http://192.168.255.60:8080"
os.environ["HTTP_PROXY"] = PROXY
os.environ["HTTPS_PROXY"] = PROXY
os.environ["http_proxy"] = PROXY
os.environ["https_proxy"] = PROXY
os.environ.pop("NO_PROXY", None)
os.environ.pop("no_proxy", None)

from qdrant_client import QdrantClient

client = QdrantClient(
    url=QDRANT_URL,          # https://...cloud.qdrant.io
    api_key=QDRANT_API_KEY,
    port=443,
    http2=False,
    timeout=30,
)

print(client.get_collections())

collections=[]


In [43]:
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    port=443,
    http2=False,  # Proxies often struggle with HTTP/2; keeping it False is safer
    timeout=30,
)

In [45]:
if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=1536,
            distance=models.Distance.COSINE
        )
    )
    print(f"Created collection {COLLECTION_NAME} with vector size 1536")

Created collection alpha-guide-rag with vector size 1536


In [46]:
vector_store = QdrantVectorStore(
    collection_name=COLLECTION_NAME,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    client=qdrant_client
)

vector_store.add_documents(split_docs)
print(f"Added {len(split_docs)} documents to {COLLECTION_NAME}")


Added 671 documents to alpha-guide-rag
